In [1]:
import pandas as pd

# ==========================================
# 1. 配置文件路径
# ==========================================
# 过滤掉转录本的
spectra_normalized = '/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.3.cNMF_NK/Example_refbuilder_NK_v2/starcat_refstarcat_consensus_spectra_score.filtered.txt'#starcat_refstarcat_consensus_spectra_normalized.filtered.txt
save_path = '/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.3.cNMF_NK/cnmf_NK_out_Sample_counts/NK_cGEP_Top30_Genes_spectra_normalized_fil.csv'  # 提取后的保存路径

# ==========================================
# 2. 读取数据矩阵
# ==========================================
print("正在读取 Spectra Score 矩阵...")
# 假设文件是制表符(\t)分隔，且第一列(index_col=0)是 cGEP 名字
df = pd.read_csv(spectra_normalized, sep='\t', index_col=0)

# ==========================================
# 3. 按行遍历提取 Top 30 基因
# ==========================================
print("正在计算并提取每个 cGEP 的 Top 30 基因...")

# 定义提取函数：对每一行(row)降序排序，取前30个的列名(index/基因名)，转为列表
def get_top_30_genes(row):
    # ascending=False 表示按分数从大到小排
    return row.sort_values(ascending=False).head(30).index.tolist()

# 使用 apply 按行(axis=1)执行该函数
top_genes_series = df.apply(get_top_30_genes, axis=1)

# ==========================================
# 4. 格式化结果并保存
# ==========================================
# 将 Series(里面是列表) 展开成 DataFrame
top_genes_df = pd.DataFrame(top_genes_series.tolist(), index=top_genes_series.index)

# 重命名列名为 Top_1, Top_2 ... Top_30
top_genes_df.columns = [f"Top_{i}" for i in range(1, 31)]

# 保存为 CSV
top_genes_df.T.to_csv(save_path)
print(f"✅ 提取成功！结果已保存至: {save_path}")

# (可选) 打印前几个看看效果
print("\n数据预览 (前 5 个 cGEP 的 Top 5 基因):")
print(top_genes_df.iloc[10:20, :3])



正在读取 Spectra Score 矩阵...
正在计算并提取每个 cGEP 的 Top 30 基因...
✅ 提取成功！结果已保存至: /sibcb1/bioinformatics/yangyue/project/immunotherapy/7.3.cNMF_NK/cnmf_NK_out_Sample_counts/NK_cGEP_Top30_Genes_spectra_normalized_fil.csv

数据预览 (前 5 个 cGEP 的 Top 5 基因):
             Top_1        Top_2       Top_3
cGEP11        CD3D         CD3G        SIT1
cGEP12     SCGB3A2  RP5-851M4.1       FABP4
cGEP13       MS4A2          HDC      TPSAB1
cGEP14       ISG15        IFIT3        IFI6
cGEP15  AC026801.2        CXCL2  AC124242.1
cGEP16        CST3         C1QC        C1QB
cGEP17   LINC01943   AC017002.3       IL2RA
cGEP18        ACTB        ACTG1        PFN1
cGEP19        CD3D         CD8B        CD8A
cGEP20    IGLV1-44      HLA-DRA    IGHV4-34


In [2]:
df.loc['cGEP34', :].sort_values(ascending=False).head(30)

LOC101928173    0.004848
MIR155HG        0.003424
TNFRSF9         0.003390
CD82            0.003330
MT1L            0.003283
LOC102724850    0.002953
CD74            0.002903
APOBEC3G        0.002835
PGAM1           0.002805
CD27            0.002779
GZMH            0.002778
PKM             0.002739
CD8A            0.002707
ZBED2           0.002705
PDCD1           0.002670
HLA-B           0.002628
LAG3            0.002590
IL2RA           0.002578
GAPDH           0.002556
GZMB            0.002504
DUSP4           0.002452
TIGIT           0.002410
LOC100996809    0.002406
VCAM1           0.002277
NKG7            0.002144
HLA-A           0.002095
LYST            0.002073
PARK7           0.002048
FKBP1A          0.002034
APOBEC3C        0.002031
Name: cGEP34, dtype: float64

In [3]:
df.loc['cGEP23', :].sort_values(ascending=False).head(30)

CRTAM         0.011341
TNFRSF9       0.010318
CCL4L2        0.010000
XCL1          0.009745
CCL4L1        0.008239
XCL2          0.008041
CCL4          0.007698
EGR2          0.006430
HBA2          0.006190
AL031666.3    0.005878
GAPDH         0.005437
RILPL2        0.005372
HBA1          0.005270
CCL3          0.004860
NFKBID        0.004836
MIR155HG      0.004826
CD82          0.004694
ZFP36L1       0.004658
SDC4          0.004648
TNF           0.004399
GEM           0.004251
PDCD1         0.004247
PKM           0.004174
GHET1         0.004085
VWDE          0.004025
EPOP          0.003861
EGR3          0.003649
AL590326.2    0.003583
FXYD2         0.003536
LINC01686     0.003521
Name: cGEP23, dtype: float64

In [4]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# ==========================================
# 1. 🌟 自定义参数设定：专属 vmax
# ==========================================
vmax_lineage    = 0.008      # 第 1 张图：Lineage
vmax_functional = 0.008    # 第 2 张图：Functional
vmax_noise      = 0.008     # 第 3 张图：Doublet_and_Artifact

# 配置文件路径
marker_info_path = '/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.3.cNMF_NK/3.5.NK.QC.MarkerGene/3.5.NK.cGEP.gene.csv'
spectra_score_path = '/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.3.cNMF_NK/Example_refbuilder_NK_v2/starcat_refstarcat_consensus_spectra_score.filtered.txt'
out_dir = '/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.3.cNMF_NK/3.5.NK.QC.MarkerGene'

vmax_dict = {
    'Lineage': vmax_lineage,
    'Functional': vmax_functional,
    'Doublet_and_Artifact': vmax_noise
}

# ==========================================
# 2. 读取数据与基础排序
# ==========================================
print("正在读取数据...")
marker_df = pd.read_csv(marker_info_path)
spectra_df = pd.read_csv(spectra_score_path, sep='\t', index_col=0)

marker_df = marker_df.dropna(subset=['Class'])

custom_class_order = ['Lineage', 'Functional', 'Doublet', 'Artifact']
marker_df['Class'] = pd.Categorical(marker_df['Class'], categories=custom_class_order, ordered=True)

# ==========================================
# 3. 智能分组
# ==========================================
def assign_plot_group(c):
    c_str = str(c).strip()
    if c_str in ['Doublet', 'Artifact']:
        return 'Doublet_and_Artifact'
    return c_str

marker_df['Plot_Group'] = marker_df['Class'].apply(assign_plot_group)
unique_groups = marker_df['Plot_Group'].dropna().unique()

print(f"检测到 {len(unique_groups)} 个绘图分组: {list(unique_groups)}")

# ==========================================
# 4. 循环生成独立热图
# ==========================================
for current_group in unique_groups:
    
    current_vmax = vmax_dict.get(current_group, 0.0075)
    print(f"\n[{current_group}] 正在绘图，当前应用阈值 vmax = {current_vmax}")
    
    group_df = marker_df[marker_df['Plot_Group'] == current_group].copy()
    group_df['sort_key'] = group_df['Cluster'].astype(str).str.extract(r'(\d+)').astype(float)
    group_df = group_df.sort_values(by=['Class', 'sort_key'])
    
    ordered_genes = []
    toshow_main = []
    ytick_labels = []
    gene_counts_per_cluster = []
    
    class_boundaries = []        
    last_class = None
    current_row_idx = 0
    
    for _, row in group_df.iterrows():
        cluster_id = row['Cluster']
        anno_name = row['Annotation']
        current_class = row['Class']
        
        if cluster_id not in spectra_df.index:
            continue
            
        if last_class is not None and current_class != last_class:
            class_boundaries.append(current_row_idx)
        last_class = current_class
            
        toshow_main.append(cluster_id)
        ytick_labels.append(f"{cluster_id}: {anno_name}")
        
        raw_genes = [g.strip() for g in str(row['MarkerGene']).replace(',', ' ').split() if g.strip()]
        
        valid_unique_genes_for_this_cgep = []
        for g in raw_genes:
            if g in spectra_df.columns and g not in valid_unique_genes_for_this_cgep:
                valid_unique_genes_for_this_cgep.append(g)
                ordered_genes.append(g)
        
        gene_counts_per_cluster.append(len(valid_unique_genes_for_this_cgep))
        current_row_idx += 1
        
    if not ordered_genes:
        print(f"⚠️ {current_group} 没有提取到有效的特异性基因，已跳过。")
        continue

    Z = spectra_df.loc[toshow_main, ordered_genes].copy()
    Z[Z > current_vmax] = current_vmax
    Z = Z.fillna(0)

    width = max(6.0, len(ordered_genes) * 0.22)
    height = max(3.5, len(toshow_main) * 0.4) 
    
    fig, ax = plt.subplots(figsize=(width, height), dpi=300)
    plt.subplots_adjust(left=0.4, bottom=0.35, right=0.95, top=0.85)

    if current_group == 'Functional':
        cbar_offset = -0.40
    elif current_group == 'Lineage':
        cbar_offset = -0.55
    else:
        cbar_offset = -0.30

    cbar_ax = ax.inset_axes([0, cbar_offset, 0.15, 0.03], transform=ax.transAxes)

    sns.heatmap(
        Z, ax=ax, vmin=0, vmax=current_vmax, cmap='mako',                
        linewidths=0, cbar_ax=cbar_ax, 
        cbar_kws={"orientation": "horizontal", 'ticks': [0, current_vmax]}, 
        xticklabels=True, yticklabels=True
    )

    ax.hlines(y=np.arange(1, len(toshow_main)), xmin=0, xmax=len(ordered_genes), 
              colors='white', linewidth=0.5, alpha=0.5)
              
    if class_boundaries:
        ax.hlines(y=class_boundaries, xmin=0, xmax=len(ordered_genes), 
                  colors='white', linewidth=2.0)

    current_x = 0
    vlines_x = []
    for count in gene_counts_per_cluster:
        current_x += count
        if count > 0: vlines_x.append(current_x)
    if vlines_x: ax.vlines(x=vlines_x[:-1], ymin=0, ymax=len(toshow_main), 
                           colors='white', linewidth=0.6, alpha=0.7)

    for _, spine in ax.spines.items():
        spine.set_visible(True); spine.set_color('white'); spine.set_linewidth(1.5)

    # ==========================================
    # 🌟 坐标轴与表头文字排版 (更新部分)
    # ==========================================
    # Y轴设置
    ax.set_yticks(np.arange(len(toshow_main)) + 0.5)
    ax.set_yticklabels(ytick_labels, fontsize=13, rotation=0)
    ax.set_ylabel('cGEP', fontsize=14, labelpad=10)  # Y轴名称

    # X轴设置
    ax.set_xticks(np.arange(len(ordered_genes)) + 0.5)
    ax.set_xticklabels(ordered_genes, fontsize=9, rotation=90) 
    ax.set_xlabel('Marker', fontsize=14, labelpad=10)  #  X轴名称
    
    # 表头修改：结合细胞类型 (current_group) 
    ax.set_title(f'{current_group} cGEP Example marker genes/proteins', fontsize=15, pad=20)
    
    # 色标设置
    cbar_ax.set_xticklabels(['0', f'{current_vmax}'])
    cbar_ax.set_title('Spectra Score', fontsize=10, pad=5)

    # 保存
    save_path = os.path.join(out_dir, f'3.5.NK_Markers_{current_group}.png')
    plt.savefig(save_path, bbox_inches="tight")
    print(f"✅ 生成成功 -> {save_path}")
    
    plt.close(fig)

print("\n🎉 表头与坐标轴名称已全部更新完毕！")

正在读取数据...
检测到 3 个绘图分组: ['Doublet_and_Artifact', 'Functional', 'Lineage']

[Doublet_and_Artifact] 正在绘图，当前应用阈值 vmax = 0.008
✅ 生成成功 -> /sibcb1/bioinformatics/yangyue/project/immunotherapy/7.3.cNMF_NK/3.5.NK.QC.MarkerGene/3.5.NK_Markers_Doublet_and_Artifact.png

[Functional] 正在绘图，当前应用阈值 vmax = 0.008
✅ 生成成功 -> /sibcb1/bioinformatics/yangyue/project/immunotherapy/7.3.cNMF_NK/3.5.NK.QC.MarkerGene/3.5.NK_Markers_Functional.png

[Lineage] 正在绘图，当前应用阈值 vmax = 0.008
✅ 生成成功 -> /sibcb1/bioinformatics/yangyue/project/immunotherapy/7.3.cNMF_NK/3.5.NK.QC.MarkerGene/3.5.NK_Markers_Lineage.png

🎉 表头与坐标轴名称已全部更新完毕！
